# Segment Analysis

Group the engineered feature datasets

- `PropertyType` / `PropertySubType`
- `CountyOrParish` / `MLSAreaMajor`
- `ListOfficeName` / `BuyerOfficeName`

In [11]:
from pathlib import Path
import pandas as pd


# scripts -> repo root
base_dir = Path.cwd().parent
processed_path = base_dir / "data" / "processed"

FILES = {
    "listings": "listings_features.csv",
    "sold": "sold_features.csv",
}

FRAMES = {
    name: pd.read_csv(processed_path / fname, low_memory=False)
    for name, fname in FILES.items()
}

for name, df in FRAMES.items():
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} cols")

listings: 466,573 rows x 43 cols
sold: 442,145 rows x 57 cols


## Metrics per dataset

In [ ]:
METRICS = {
    "listings": ["ListPrice", "OriginalListPrice", "LivingArea", "days_on_market"],
    "sold": [
        "ClosePrice", "OriginalListPrice", "ListPrice", "LivingArea",
        "price_ratio", "price_per_sqft", "days_on_market",
        "listing_to_contract_days", "contract_to_close_days",
    ],
}

# keep only metrics that exist 
METRICS = {
    name: [c for c in cols if c in FRAMES[name].columns]
    for name, cols in METRICS.items()
}

## Segment summary 

reports `count`, `mean`, `median`, `std`,
`min`, `max` for each metric.

In [ ]:
SEGMENTS = [
    ("PropertyType", "PropertySubType"),
    ("CountyOrParish", "MLSAreaMajor"),
    ("ListOfficeName", "BuyerOfficeName"),
]


def segment_summary(df, group_cols, metrics, top=20):
    group_cols = [c for c in group_cols if c in df.columns]
    if not group_cols or not metrics:
        return None

    summary = df.groupby(group_cols, dropna=False)[metrics].agg(
        ["count", "mean", "median", "std", "min", "max"]
    )
    summary = summary.sort_values((metrics[0], "count"), ascending=False).head(top)
    return summary.reset_index().round(2)

In [14]:
for name, df in FRAMES.items():
    for group_cols in SEGMENTS:
        print("=" * 60)
        print(f"{name} — {' x '.join(group_cols)}")
        print("=" * 60)
        summary = segment_summary(df, group_cols, METRICS[name])
        if summary is None:
            print("(skipped — no grouping columns present)")
        else:
            display(summary)

listings — PropertyType x PropertySubType


PropertyType        PropertySubType ListPrice                         \
                                           count        mean     median   
0   Residential  SingleFamilyResidence    340940  1492878.10   899999.0   
1   Residential            Condominium     84338   813711.42   639000.0   
2   Residential              Townhouse     26123   940547.11   799000.0   
3   Residential     ManufacturedOnLand      6823   375674.32   329990.0   
4   Residential                 Duplex      2977  1335944.81   974000.0   
5   Residential       StockCooperative      1572   416292.42   375000.0   
6   Residential                    NaN       949  1253859.06   799000.0   
7   Residential                  Cabin       899   369124.53   299000.0   
8   Residential                Triplex       631  1489315.38  1199000.0   
9   Residential               MixedUse       463  1540025.52   850000.0   
10  Residential             Quadruplex       245  1754987.47  1469000.0   
11  Residential             MobileHome       148   384354.90   270950.0   
12  Residential               BoatSlip       114   257760.51   230000.0   
13  Residential             OwnYourOwn        91   851087.73   345000.0   
14  Residential                   Loft        54   661957.39   677000.0   
15  Residential            CoOwnership        51   863517.39   525000.0   
16  Residential              Timeshare        42   200819.05    19000.0   
17  Residential       ManufacturedHome        41   488302.12   349950.0   
18  Residential                   Farm        33  2901750.73  2312336.0   
19  Residential                 Studio        33   430208.79   435000.0   

                                      OriginalListPrice              \
           std       min          max             count        mean   
0   2843331.35     100.0  400000000.0            340498  1578903.59   
1    810602.21    1500.0   39500000.0             84264   898119.59   
2    514229.86   82500.0   14150000.0             26093  1022458.27   
3    315669.21    7500.0   13000000.0              6823   381758.83   
4   1274003.19   26000.0   15999000.0              2977  1490389.86   
5    225049.96  125000.0    3895000.0              1572   420220.71   
6   1897907.36   35000.0   24950000.0               947  1887011.00   
7    403971.32   14995.0    5000000.0               899   379405.36   
8   1172681.69  149000.0    8995000.0               631  1498058.11   
9   4363062.68   28888.0   85000000.0               463  1796308.63   
10  1693631.81  248000.0   14950000.0               245  1757305.65   
11   311168.07   25000.0    1395000.0               148   388938.34   
12   151738.33   26800.0    1400000.0               114   274585.95   
13  1920874.27   29996.0   10000000.0                91   855967.99   
14   232613.33  300000.0    1599000.0                54   666677.76   
15  1196504.79  162500.0    6499000.0                51   795693.86   
16   304794.78    2000.0    1250000.0                42   201176.19   
17   633776.20   99000.0    3890000.0                41   489036.27   
18  2661927.86  650000.0   14998000.0                31  3043175.42   
19   142533.89   32800.0     699900.0                33   435666.36   

                                                    LivingArea           \
       median          std        min           max      count     mean   
0    910000.0   7728715.48       0.00  1.390000e+09     340528  2167.69   
1    645000.0   6707287.41       1.00  8.195000e+08      84287  1271.04   
2    799900.0   7071487.68       1.30  8.299000e+08      26115  2192.76   
3    335000.0    350462.53     185.00  1.300000e+07       6823  1439.56   
4    975000.0   5916657.78       1.08  2.999000e+08       2977  1976.63   
5    375000.0    227625.19     350.00  3.895000e+06       1572   962.49   
6    799900.0  19517079.52    2700.00  5.990000e+08        942  2060.85   
7    299999.0    431502.46   12900.00  5.000000e+06        899   973.24   
8   1199900.0   1185941.46   30000.00  

listings — CountyOrParish x MLSAreaMajor


CountyOrParish                          MLSAreaMajor ListPrice  \
                                                             count   
0        Riverside    SRCAR - Southwest Riverside County     21603   
1      Santa Clara                     699 - Not Defined     16450   
2     Contra Costa                                   NaN     16026   
3          Alameda                                   NaN     15318   
4        San Mateo                     699 - Not Defined      6372   
5        Riverside                       252 - Riverside      5519   
6            Butte                                   NaN      4639   
7      Los Angeles                       LAC - Lancaster      3735   
8        Riverside                          248 - Corona      3501   
9         Monterey                     699 - Not Defined      3417   
10       Riverside      313 - La Quinta South of HWY 111      3386   
11  San Bernardino                     VIC - Victorville      3329   
12       Riverside  263 - Banning/Beaumont/Cherry Valley      3247   
13       Riverside                     699 - Not Defined      3111   
14  San Bernardino                  274 - San Bernardino      3014   
15      Santa Cruz                     699 - Not Defined      3011   
16     Los Angeles                        PLM - Palmdale      2895   
17  San Bernardino                         686 - Ontario      2839   
18       Riverside                   321 - Rancho Mirage      2821   
19       Riverside                   259 - Moreno Valley      2746   

                                                              \
          mean     median         std       min          max   
0    657460.81   598888.0   417704.73   49900.0   11900000.0   
1   1945343.85  1500000.0  1656661.13  220000.0   39988000.0   
2   1115948.30   799999.0   913138.56  110000.0   25000000.0   
3   1176934.91   998000.0   743974.85   99800.0   14800000.0   
4   2432044.68  1688000.0  3539170.74    3500.0  125000000.0   
5    754051.73   675000.0   357177.41   79900.0    6400000.0   
6    454699.93   405000.0   290694.78   25000.0    9900000.0   
7    503335.65   480000.0   161938.89   75000.0    2500000.0   
8    845600.83   775000.0   401523.80  174900.0    6300000.0   
9   1641432.34   965000.0  3295490.77  111641.0  100000000.0   
10  1573288.74   948750.0  2827972.48   65000.0   85000000.0   
11   451534.56   449000.0    95014.13   99000.0    1200000.0   
12   506840.15   499900.0   198775.81   59000.0    4950000.0   
13   624760.96   565000.0   422838.91   64900.0   10000000.0   
14   518571.73   499999.0   159855.53   75000.0    1888888.0   
15  1480016.15  1200000.0  1205078.15  179000.0   14950000.0   
16   560808.86   529000.0   177903.01  100000.0    2995000.0   
17   676661.75   659900.0   220423.81  120000.0    7840000.0   
18  1418486.07   975000.0  1380951.40  109000.0   19000000.0   
19   570783.68   560000.0   111793.22  180000.0    1999999.0   

   OriginalListPrice                                                 \
               count        mean     median          std        min   
0              21603   670345.08   599000.0    769123.77       1.00   
1              16212  1949490.40  1520000.0   1663691.23  220000.00   
2              16026  1529261.16   819000.0  15874809.57  110000.00   
3              15318  1588488.97   998000.0  16309593.61    1275.00   
4               6224  2436647.06  1688000.0   3572571.37    3500.00   
5               5519   768242.80   676990.0    470440.36     739.00   
6               4638   459595.50   411000.0    296740.82     317.00   
7               3735   510969.66   486000.0    247989.80      45.00   
8               3501   855523.65   777000.0    487521.47       1.12   
9               3355  1656311.22   975000.0   3322917.35  180688.00   
10              3386  1592352.35   949000.0   2872476.86     595.00   
11              3329   589679.09   449900.0   7706224.92     599.00   
12              3247   512166.09   499999.0    225997.49    

listings — ListOfficeName x BuyerOfficeName


...

PropertyType        PropertySubType ClosePrice                         \
                                            count        mean     median   
0   Residential  SingleFamilyResidence     331099  1281868.62   880000.0   
1   Residential            Condominium      72975   875170.26   626000.0   
2   Residential              Townhouse      25526  1008894.57   800000.0   
3   Residential     ManufacturedOnLand       5859   352156.67   323000.0   
4   Residential                 Duplex       2445  1221168.55   910000.0   
5   Residential       StockCooperative       1842   391839.87   360000.0   
6   Residential                    NaN        789  1002856.23   810000.0   
7   Residential                  Cabin        512   273590.13   225000.0   
8   Residential                Triplex        366  1311013.23  1102500.0   
9   Residential               MixedUse        226  1016257.76   705000.0   
10  Residential             Quadruplex        154  1433663.65  1250500.0   
11  Residential               BoatSlip         89   212368.54   185000.0   
12  Residential             OwnYourOwn         65   334172.40   272000.0   
13  Residential             MobileHome         57   474701.75   500000.0   
14  Residential       ManufacturedHome         37   302882.65   300000.0   
15  Residential                   Loft         30   701426.30   687500.0   
16  Residential              Timeshare         20   414225.00   242500.0   
17  Residential            CoOwnership         19   730236.79   400000.0   
18  Residential                   Farm         14  1765642.86  1500000.0   
19  Residential                 Studio         12   342540.83   319000.0   

                                      OriginalListPrice              \
           std       min          max             count        mean   
0   5468299.51       0.0  989500000.0            330518  1330233.84   
1   8514881.48     464.0  890000000.0             72880   830394.74   
2   6882692.75     675.0  850000000.0             25484   986585.77   
3    223921.68     500.0    5900000.0              5859   372199.01   
4   1092463.89   35000.0   14000000.0              2445  1282400.54   
5    188558.93  110000.0    3690000.0              1842   405022.83   
6    735059.51   18000.0    7500000.0               787  2097012.95   
7    273243.90   14000.0    2989000.0               512   314288.85   
8    955197.60  120000.0    7100000.0               366  1383625.65   
9   1211852.17   17300.0   11195000.0               226  1683177.89   
10  1136645.37  285000.0   11995000.0               154  1542422.90   
11    78155.51  100000.0     450000.0                89   249234.82   
12   223843.01  129900.0    1400000.0                65   357701.48   
13   289409.40   22000.0    1350000.0                57   523826.32   
14   124916.60   99000.0     597000.0                37   319628.30   
15   194030.33  316000.0    1315000.0                30   716929.97   
16   410563.75    4500.0    1225000.0                20   424095.00   
17  1118082.85  220000.0    5250000.0                19   582359.99   
18  1109200.17  490000.0    4200000.0                13  1783146.08   
19    95442.79  250000.0     590000.0                12   360249.17   

                                                    ListPrice              \
       median          std        min           max     count        mean   
0    890000.0   6868727.67       0.00  1.390000e+09    331101  1244749.08   
1    639900.0   4870199.57       1.00  7.489500e+08     72975   771375.15   
2    799000.0   4974350.19       1.30  5.898880e+08     25526   925869.46   
3    340000.0    247800.63     185.00  5.995000e+06      5859   359785.39   
4    930000.0   1224783.98       1.08  1.400000e+07      2445  1241875.76   
5    370000.0    207887.84     350.00  3.895000e+06      1842   396690.00   
6    839500.0  23118441.29   35000.00  5.990000e+08       789  1002832.87   
7    264900.0    363799.32   12900.00  4.399999e+06       512   288388.71 

sold — CountyOrParish x MLSAreaMajor


CountyOrParish                          MLSAreaMajor ClosePrice  \
                                                              count   
0        Riverside    SRCAR - Southwest Riverside County      22266   
1     Contra Costa                                   NaN      17571   
2          Alameda                                   NaN      17174   
3      Santa Clara                     699 - Not Defined      16496   
4        San Mateo                     699 - Not Defined       6573   
5        Riverside                       252 - Riverside       5792   
6            Butte                                   NaN       4423   
7        Riverside                          248 - Corona       3727   
8         Monterey                     699 - Not Defined       3708   
9      Los Angeles                       LAC - Lancaster       3415   
10  San Bernardino                     VIC - Victorville       3278   
11  San Bernardino                  274 - San Bernardino       3215   
12       Riverside  263 - Banning/Beaumont/Cherry Valley       3179   
13       Riverside                   259 - Moreno Valley       2910   
14  San Bernardino                         686 - Ontario       2869   
15  San Bernardino                         264 - Fontana       2809   
16       Riverside                     699 - Not Defined       2765   
17      Santa Cruz                     699 - Not Defined       2755   
18  San Bernardino                688 - Rancho Cucamonga       2713   
19          Merced                                   NaN       2643   

                                                              \
          mean     median         std       min          max   
0    636331.83   585000.0  3292070.88   15000.0  489900000.0   
1   1140024.70   825229.0  4597815.54     345.0  600000000.0   
2   1316886.96  1134000.0  5926283.10     464.0  765000000.0   
3   1956199.84  1620000.0  1309789.68  215000.0   21250000.0   
4   2258150.81  1750000.0  2015669.15  138500.0   31800000.0   
5    708962.51   655000.0   259647.91   35000.0    4900202.0   
6    431959.45   399000.0   232593.28   15000.0    5550000.0   
7    795607.54   755000.0   275548.16  174900.0    3500000.0   
8   1393023.75   910000.0  1570351.24  111641.0   28000000.0   
9    489696.91   475000.0   227630.47   35000.0   11000000.0   
10   579608.19   438000.0  8236818.96   41500.0  472000000.0   
11   504646.47   495000.0   201634.70   44500.0    6480000.0   
12   492068.68   499000.0   175296.80   60000.0    6400043.0   
13   557378.65   555000.0    97379.53  180000.0    1390000.0   
14   665373.58   652990.0   214067.59  189900.0    8180000.0   
15   656701.71   648000.0   135677.24  223000.0    1455000.0   
16   574940.94   545000.0   320282.46   55000.0    7400000.0   
17  1346705.79  1195000.0   879182.51   75000.0   12578750.0   
18   857803.15   772000.0   358595.06  115000.0    5200000.0   
19   437973.31   410000.0   167447.19   30300.0    3090000.0   

   OriginalListPrice                                                 \
               count        mean     median          std        min   
0              22266   635204.22   590000.0    715002.89       1.00   
1              17570  1417631.78   825000.0  13120731.36  110000.00   
2              17172  1528049.85  1025000.0  14979623.22     950.00   
3              16204  1877150.37  1500000.0   1304824.69  220000.00   
4               6364  2209327.70  1688000.0   2156229.68  138500.00   
5               5792   735828.99   660000.0    600792.53    8249.00   
6               4422   448160.99   409000.0    237256.09     110.00   
7               3727   812003.05   760000.0    337271.47       1.12   
8               3622  1461141.81   939999.5   1743956.85  180688.00   
9               3415   494356.01   479900.0    148396.68     470.00   
10              3278   444573.09   439900.0    161253.20     325.00   
11              3215   507204.87   498888.0    172153.31     360.00   
12              3179   666377.44   499

sold — ListOfficeName x BuyerOfficeName


ListOfficeName  \
                                                        
0                                             Compass   
1                              Coldwell Banker Realty   
2                              Coldwell Banker Realty   
3                              Keller Williams Realty   
4                                             Compass   
5                              First Team Real Estate   
6   Berkshire Hathaway HomeServices California Pro...   
7                         Intero Real Estate Services   
8                                             Compass   
9                                        Equity Union   
10                                        Real Broker   
11                       eXp Realty of California Inc   
12                             Coldwell Banker Realty   
13                        Intero Real Estate Services   
14                     eXp Realty of California, Inc.   
15                                 Century 21 Masters   
16  Berkshire Hathaway HomeServices California Pro...   
17                              Bennion Deville Homes   
18                               Coldwell Banker West   
19                           Seven Gables Real Estate   

                                      BuyerOfficeName ClosePrice              \
                                                           count        mean   
0                                             Compass       7670  2128263.20   
1                              Coldwell Banker Realty       3699  2047259.88   
2                                             Compass       1648  1988369.36   
3                              Keller Williams Realty       1491  1073664.54   
4                              Coldwell Banker Realty       1350  2220190.99   
5                              First Team Real Estate       1255  1245762.84   
6   Berkshire Hathaway HomeServices California Pro...        859  1660929.84   
7                         Intero Real Estate Services        832  1731741.17   
8                                                 NaN        821  1806780.52   
9                                        Equity Union        813  1042128.43   
10                                        Real Broker        758  1141720.00   
11                       eXp Realty of California Inc        720  1053966.42   
12                                                NaN        644  1477845.66   
13                                                NaN        606  1342388.48   
14                     eXp Realty of California, Inc.        579   938761.22   
15                                     NONMEMBER MRML        537   609472.16   
16                                            Compass        510  2044909.72   
17                              Bennion Deville Homes        509   967649.61   
18                               Coldwell Banker West        497  4430096.62   
19                           Seven Gables Real Estate        471  1245490.57   

                                                  OriginalListPrice  \
       median          std       min          max             count   
0   1585000.0   3002182.19  130000.0  199947850.0              7583   
1   1300000.0  14059990.89   68500.0  850000000.0              3653   
2   1496500.0   1690812.56   85000.0   16000000.0              1637   
3    910000.0    667708.47  135000.0    8800000.0              1486   
4   1600000.0   2723735.46  305000.0   38000000.0              1342   
5   1100000.0    816036.94   35000.0   13888800.0              1255   
6   1050000.0   2000790.83  165000.0   32000000.0               859   
7   1456000.0   1080580.14  335000.0    7888520.0               812   
8   1410000.0   1802997.69  149000.0   23000000.0               808   
9    860000.0    745427.79   13000.0    8920000.0               813   
10   865000.0    833494.35   99000.0    5620000.0               758   
11   807500.0    906617.93   26000.0   10099000.0               713   
12  1221250.0   1745099.78  1